<a href="https://colab.research.google.com/github/appolinecontact-gif/AI-character/blob/main/character-consistency/facial_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛠️ 1. Install Dependencies & Setup
---

In [ ]:
!pip install -q insightface onnxruntime-gpu opencv-python numpy

<div align="center">
  <h1>🖼️ Image Comparison</h1>
</div>

In [ ]:
import cv2
import numpy as np
import os
import glob
from insightface.app import FaceAnalysis

# 1. Setup
app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

def get_embed(img_path):
    img = cv2.imread(img_path)
    if img is None: return None
    faces = app.get(img)
    return faces[0].normed_embedding if len(faces) > 0 else None

# 2. File Discovery (Sidebar Mode)
ref_files = glob.glob("ref.*")
reference_image = ref_files[0] if ref_files else None
all_images = glob.glob("*.jpg") + glob.glob("*.png") + glob.glob("*.jpeg")
test_images = [img for img in all_images if img != reference_image]

# 3. Execution with Strict Scoring
if reference_image:
    print(f"--- Strict Consistency Report ---")
    print(f"Ref: {reference_image}\n")

    ref_feat = get_embed(reference_image)

    if ref_feat is not None:
        print(f"{'IMAGE':<20} | {'SCORE':<8} | {'VERDICT'}")
        print("-" * 50)

        for img_path in sorted(test_images):
            test_feat = get_embed(img_path)

            if test_feat is not None:
                # Pure Cosine Similarity (Your original math)
                sim = np.dot(ref_feat, test_feat)

                # Simple Linear Scale (Strict 0-10)
                # If sim is 0.65, score is 6.5
                final_score = round(sim * 10, 1)
                final_score = min(max(final_score, 0), 10)

                # New parameters based on your strict preference
                if final_score >= 7.0:
                    verdict = "✅ SAME CHARACTER"
                elif final_score >= 5.0:
                    verdict = "⚠️ IDENTITY DRIFT"
                else:
                    verdict = "❌ NOT A MATCH"

                print(f"{img_path:<20} | {final_score:<8} | {verdict}")
            else:
                print(f"{img_path:<20} | No Face Detected")
    else:
        print("Error: Could not extract features from reference.")
else:
    print("Error: Rename your original image to 'ref.jpg' in the sidebar.")